# Praktikum Pengenalan Vowel Bahasa Indonesia dengan DTW dan Model Gaussian

**Mata kuliah:** IF4071 Pemrosesan Ucapan

**Kelompok:** Kelompok 7

**Anggota:** 
- Muhammad Alfansya	(13523005)
- Shannon Aurellius Anastasya Lie (13523019)
- Dzaky Aurelia Fawwaz (13523065)
- Ferdin Arsenarendra Purtadi (13523117)
- Naomi Risaka Sitorus (13523122)

Notebook ini ...

## 1. Instalasi Library

Jalankan jika library belum tersedia pada environment notebook.

In [1]:
%pip install numpy pandas matplotlib seaborn soundfile scipy python_speech_features scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Import Library & Konfigurasi

In [2]:
from pathlib import Path
from math import gcd
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf

from scipy.signal import resample_poly
from python_speech_features import mfcc, delta
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
DATA_DIR = Path("data")
VOWELS = ["A", "I", "U", "E", "O"]
SAMPLE_RATE = 16000
WINLEN = 0.025
WINSTEP = 0.010
NUM_CEP = 13
N_FFT = 512
COVARIANCE_REG = 1e-3
TARGET_TEMPLATE_FRAMES = 100

np.random.seed(SEED)

print("Dataset directory:", DATA_DIR.resolve())
print("Directory exists:", DATA_DIR.exists())

Dataset directory: C:\Users\naomi\Downloads\Pengenalan_Vowel_Bahasa_Indonesia\data
Directory exists: True


## 3. Import Dataset

Penamaan file dilakukan sesuai konvensi: S{speaker}_{vowel}_{usage}_{condition}.wav

Usage: CR (closed recording), CS (closed speaker), atau OS (open speaker)

Condition: N (normal), HP (high pitch), LP (low pitch), SD (short duration), atau LD (long duration)

## 4. Validasi File Audio

Periksa apakah lima vowel tersedia pada data template, apakah file audio bisa dibaca, serta apakah sample rate dan durasinya masuk akal.

## 5. Preprocessing Audio

## 6. Ekstraksi MFCC 39 Dimensi

## 7. Visualisasi Contoh Rekaman Pasca Ekstraksi

Plot waveform dan jumlah frame fitur untuk memeriksa bahwa proses loading serta ekstraksi berjalan.

## 8. Averaged Template untuk DTW

Urutan fitur dari setiap speaker dapat memiliki jumlah frame berbeda sehingga perlu dibuat averaged template. Template setiap vowel dibangun hanya dari file pada folder `template`.


In [ ]:
from IPython.display import display

from src.run_experiments import scan_dataset
from src.template import TARGET_TEMPLATE_FRAMES as TEMPLATE_FRAMES, build_templates, resample_frames

try:
    from src.features import extract_mfcc39
except ImportError:
    from src._stubs import extract_mfcc39

assert TEMPLATE_FRAMES == TARGET_TEMPLATE_FRAMES, "Samakan TARGET_TEMPLATE_FRAMES di notebook dan src/template.py"

template_files = scan_dataset(DATA_DIR).query("scenario == 'template'")
template_feats = {
    v: [extract_mfcc39(path) for path in template_files[template_files.vowel == v].path] for v in VOWELS
}

templates = build_templates(template_feats)

summary = pd.DataFrame({
    v: {
        "jumlah rekaman": len(fs),
        "frame min": min(len(f) for f in fs),
        "frame median": int(np.median([len(f) for f in fs])),
        "frame max": max(len(f) for f in fs),
        "frame template": len(templates[v]),
    }
    for v, fs in template_feats.items()
}).T
display(summary)

fig, axes = plt.subplots(1, len(VOWELS), figsize=(3.2 * len(VOWELS), 2.8), sharey=True)
for ax, v in zip(axes, VOWELS):
    for f in template_feats[v]:
        ax.plot(resample_frames(f, TEMPLATE_FRAMES)[:, 1], color="gray", alpha=0.5, lw=1)
    ax.plot(templates[v][:, 1], color="C3", lw=2)
    ax.set_title(f"Vowel {v}")
    ax.set_xlabel("frame")
axes[0].set_ylabel("MFCC ke-2 (ternormalisasi)")
plt.tight_layout()
plt.show()

## 9. Implementasi DTW

DTW menghitung akumulasi jarak frame dengan dynamic programming. Biaya akhir dinormalisasi terhadap panjang path agar perbandingan urutan tidak terlalu dipengaruhi jumlah langkah alignment. Input diprediksi sebagai vowel dengan biaya terkecil.


In [ ]:
from IPython.display import display

from src.dtw import dtw_distance, dtw_path, local_cost

X = template_feats["A"][0]
assert dtw_distance(X, X) == 0.0, "jarak DTW ke dirinya sendiri harus 0"
assert np.isclose(dtw_distance(X, templates["A"]), dtw_distance(templates["A"], X)), "DTW harus simetris"
cost, path = dtw_path(X, templates["A"])
assert np.isclose(cost, dtw_distance(X, templates["A"])), "dtw_path dan dtw_distance harus sama"
print(f"Rekaman A pertama ke template A: cost {cost:.3f}, panjang path {len(path)} sel")

cost_table = pd.DataFrame(
    {tv: {v: dtw_distance(template_feats[v][0], templates[tv]) for v in VOWELS} for tv in VOWELS}
)
cost_table["prediksi"] = cost_table[VOWELS].idxmin(axis=1)
cost_table.index.name = "rekaman"
cost_table.columns.name = "template"
display(cost_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, v in zip(axes, ["A", "I"]):
    c, p = dtw_path(X, templates[v])
    ax.imshow(local_cost(X, templates[v]).T, origin="lower", aspect="auto", cmap="viridis")
    ax.plot(p[:, 0], p[:, 1], color="white", lw=1.5)
    ax.set_title(f"Rekaman A vs template {v} (cost {c:.2f})")
    ax.set_xlabel("frame rekaman")
    ax.set_ylabel("frame template")
plt.tight_layout()
plt.show()

## 10. Implementasi Model Gaussian

Baseline Gaussian di sini menggabungkan frame MFCC dari seluruh rekaman template pada kelas vowel yang sama. Setiap kelas memiliki mean dan covariance 39 × 39. Covariance diberi regularisasi diagonal agar lebih stabil secara numerik. Saat pengujian, negative log-likelihood dihitung untuk setiap frame dan dirata-ratakan sepanjang rekaman. Model dengan nilai terkecil menjadi prediksi.


## 11. Persiapan Skenario Pengujian

Template DTW dan model Gaussian hanya dibangun dari folder `data/template`, lalu diuji pada skenario berikut:

- **Closed recordings** (`data/test/closed/recording`): file template yang sama diuji kembali. Hasilnya hanya diagnostik karena model telah dibangun menggunakan rekaman tersebut.
- **Closed speakers** (`data/test/closed/speaker`): rekaman lain dari speaker yang juga terdapat pada data template, dalam lima kondisi (N, HP, LP, SD, LD).
- **Open speakers** (`data/test/open`): rekaman dari speaker di luar data template, yaitu anggota kelompok lain.
- *(Opsional)* **Leave-one-speaker-out** (`loso`): template dibangun dari 4 speaker, lalu diuji pada data closed speaker milik speaker ke-5, bergiliran untuk setiap speaker.

Sebelum pengujian, `check_dataset` memastikan pengujian adil dan label benar: nama file sesuai konvensi, label folder sama dengan label nama file, tidak ada file identik dengan label berbeda, closed recording identik dengan template, closed speaker tidak memuat rekaman template, speaker open bukan speaker template, dan tidak ada rekaman yang belum dipotong (durasi > 10 detik). Jika ada pelanggaran, eksekusi berhenti dengan pesan yang menyebutkan file bermasalah.

In [ ]:
# Jika file di src/ diubah, restart kernel agar perubahan terbaca
from src.run_experiments import SCENARIO_DIRS, check_dataset, run_all, scan_dataset

dataset = scan_dataset(DATA_DIR)
for warning in check_dataset(dataset):
    print("[peringatan]", warning)

# Jumlah file per skenario dan vowel
pd.crosstab(dataset.scenario, dataset.vowel, margins=True, margins_name="Total").reindex(
    index=list(SCENARIO_DIRS) + ["Total"], columns=VOWELS + ["Total"], fill_value=0
)

In [ ]:
# Speaker dan kondisi rekaman pada setiap skenario
join_unique = lambda s: ", ".join(sorted(s.dropna().unique())) or "-"
dataset.groupby("scenario").agg(
    jumlah_file=("file", "size"),
    speaker=("speaker", join_unique),
    kondisi=("condition", join_unique),
).reindex(list(SCENARIO_DIRS)).fillna({"jumlah_file": 0, "speaker": "-", "kondisi": "-"}).astype({"jumlah_file": int})

## 12. Pengujian

`run_all` menjalankan seluruh alur pengujian: ekstraksi MFCC setiap file, membangun template DTW dan model Gaussian sekali dari data template, lalu mengenali setiap file uji pada setiap skenario. Prediksi adalah vowel dengan skor terkecil (cost DTW atau negative log-likelihood).

Hasil disimpan ke `results/predictions.csv`, dengan **satu baris per file uji × sistem**:

| Kolom | Isi |
|---|---|
| `file`, `speaker`, `condition` | identitas rekaman uji |
| `scenario` | `closed_recording`, `closed_speaker`, `open`, atau `loso` |
| `system` | `dtw`, `gaussian`, atau `dtw_nn` (DTW ke setiap template individual, tanpa dirata-rata) |
| `vowel_true`, `pred`, `correct` | label sebenarnya, hasil prediksi, dan 1 jika benar |
| `score_A` … `score_O` | skor ketidakmiripan terhadap setiap vowel (makin kecil makin mirip) |
| `margin` | selisih skor terbaik kedua dengan skor terbaik (makin besar makin yakin) |
| `n_frames`, `time_ms` | jumlah frame MFCC dan waktu recognition |

File ini menjadi sumber seluruh tabel akurasi dan confusion matrix pada bagian selanjutnya. Konfigurasi run dicatat di `results/run_info.json`.

`USE_STUBS = True` memakai implementasi sementara di `src/_stubs.py`. Ubah menjadi `False` setelah kode `features.py`, `template.py`, `dtw.py`, dan `gaussian.py` selesai.

In [ ]:
USE_STUBS = True                                            # False setelah kode P1-P3 siap
SCENARIOS = ["closed_recording", "closed_speaker", "open"]  # opsional: tambahkan "loso"
SYSTEMS = ["dtw", "gaussian"]                               # opsional: tambahkan "dtw_nn"

results = run_all(DATA_DIR, "results", scenarios=SCENARIOS, systems=SYSTEMS, use_stubs=USE_STUBS)
results.head(10)

In [ ]:
# Jumlah prediksi per skenario dan sistem
results.pivot_table(index="scenario", columns="system", values="file", aggfunc="size")

In [ ]:
# Prediksi salah, diurutkan dari yang paling yakin (margin terbesar)
results.loc[results.correct == 0, ["file", "scenario", "system", "vowel_true", "pred", "margin"]].sort_values(
    "margin", ascending=False
).head(10)

## 13. Akurasi Hasil Pengujian

### A. Akurasi Keseluruhan

Akurasi = jumlah prediksi benar / jumlah seluruh data uji × 100%. Nilai rata-rata pada tabel dihitung dari skenario yang memiliki data hasil.

### B. Akurasi per Vowel

Akurasi per vowel dihitung sebagai proporsi rekaman dari vowel tersebut yang berhasil dikenali dengan benar untuk masing-masing sistem dan skenario.

### C. Akurasi per Speaker

Hasil tiap speaker dihitung untuk masing-masing sistem dan skenario.

## 14. Confusion Matrix

Confusion matrix membantu melihat vowel mana yang sering tertukar. Baris merupakan label sebenarnya dan kolom merupakan hasil prediksi.


## 15. Ringkasan Hasil Pengujian


## 17. Export Hasil Pengujian

Hasil disimpan sebagai CSV agar bisa dipakai bersama untuk laporan dan pemeriksaan hasil.
